# Training CNEP/CNMP with Tactile Data - Example Notebook

This notebook demonstrates how to use tactile images and 3D force as conditional inputs for CNEP and CNMP models.

## Setup

First, we'll import the necessary modules and check our environment.

In [ ]:
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import os

# Add paths
sys.path.append('../models/')
sys.path.append('../data/')

from cnep import CNEP
from cnmp import CNMP
from tactile_data_loader import TactileDataset, create_feature_extractor

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Step 1: Generate Example Data

If you don't have your own tactile data yet, let's generate some synthetic data for testing.

In [ ]:
from generate_example_tactile_data import generate_synthetic_tactile_data

# Generate synthetic data
data_folder = '../data/example_tactile_data'
num_demos = 20
num_timesteps = 200

if not os.path.exists(data_folder):
    print("Generating synthetic tactile data...")
    generate_synthetic_tactile_data(
        output_folder=data_folder,
        num_demos=num_demos,
        num_timesteps=num_timesteps,
        img_size=(224, 224)
    )
else:
    print(f"Data already exists at {data_folder}")

## Step 2: Load Data

Now let's load the tactile dataset using the TactileDataset class.

In [ ]:
# Create feature extractor
print("Creating feature extractor...")
feature_extractor = create_feature_extractor('mobilenet_v2', device=device)

# Load dataset
print("Loading dataset...")
dataset = TactileDataset(
    data_folder=data_folder,
    demo_folders=[f'demo_{i}' for i in range(num_demos)],
    img_subfolder='img',
    csv_filename='data.csv',
    trajectory_cols=[0, 1, 2],
    force_cols=[3, 4, 5],
    num_timesteps=num_timesteps,
    normalize=True,
    img_feature_extractor=feature_extractor,
    device=device
)

# Get all data
trajectories, forces, img_features = dataset.get_all_data()

print(f"\nDataset loaded:")
print(f"  Number of demonstrations: {len(dataset)}")
print(f"  Trajectory shape: {trajectories.shape}")
print(f"  Force shape: {forces.shape}")
print(f"  Image features shape: {img_features.shape}")

## Step 3: Visualize Data

Let's visualize some of the loaded trajectories.

In [ ]:
# Denormalize for visualization
trajs_vis = dataset.denormalize_trajectory(trajectories[:5])
forces_vis = dataset.denormalize_force(forces[:5])

fig = plt.figure(figsize=(15, 5))

# Plot trajectories
ax1 = fig.add_subplot(121, projection='3d')
for i in range(5):
    traj = trajs_vis[i].cpu().numpy()
    ax1.plot(traj[:, 0], traj[:, 1], traj[:, 2], label=f'Demo {i}')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title('Robot Trajectories')
ax1.legend()

# Plot forces
ax2 = fig.add_subplot(122)
for i in range(5):
    force = forces_vis[i].cpu().numpy()
    force_mag = np.sqrt(np.sum(force**2, axis=1))
    ax2.plot(force_mag, label=f'Demo {i}')
ax2.set_xlabel('Timestep')
ax2.set_ylabel('Force Magnitude')
ax2.set_title('Force Profiles')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Step 4: Split Data

Split the data into training and validation sets.

In [ ]:
# Split data
train_ratio = 0.8
num_train = int(len(dataset) * train_ratio)

perm_ids = torch.randperm(len(dataset))
train_ids = perm_ids[:num_train]
val_ids = perm_ids[num_train:]

train_trajs = trajectories[train_ids]
train_forces = forces[train_ids]
train_img_feats = img_features[train_ids]

val_trajs = trajectories[val_ids]
val_forces = forces[val_ids]
val_img_feats = img_features[val_ids]

print(f"Train: {len(train_ids)} demos")
print(f"Val: {len(val_ids)} demos")

## Step 5: Create Model

Create a CNEP model for training.

In [ ]:
# Dimensions
dx = 1  # Time
dy = 3  # Trajectory (x, y, z)
df = 3  # Force (fx, fy, fz)
dg = img_features.shape[-1]  # Image features
d_cond = df + dg  # Total conditioning

print(f"Dimensions:")
print(f"  Time (dx): {dx}")
print(f"  Trajectory (dy): {dy}")
print(f"  Force (df): {df}")
print(f"  Image features (dg): {dg}")
print(f"  Total conditioning: {d_cond}")

# Create CNEP model
batch_size = 4
model = CNEP(
    input_dim=dx + d_cond,
    output_dim=dy,
    n_max=20,
    m_max=20,
    encoder_hidden_dims=[256, 256],
    num_decoders=2,
    decoder_hidden_dims=[128, 128],
    batch_size=batch_size,
    scale_coefs=True,
    device=device
)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel created with {count_parameters(model):,} parameters")

## Step 6: Training Loop (Simplified)

A simplified training loop for demonstration. For full training, use the provided scripts.

In [ ]:
# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
mse_loss = torch.nn.MSELoss()

# Move data to device
train_trajs = train_trajs.to(device)
train_forces = train_forces.to(device)
train_img_feats = train_img_feats.to(device)

print("Starting simplified training (100 epochs)...")
print("For full training, use train_cnep_on_tactile.py")

losses = []
for epoch in range(100):
    # Sample a batch
    batch_ids = torch.randperm(len(train_ids))[:batch_size]
    
    # Prepare simple batch (non-masked for simplicity)
    obs = torch.cat([
        torch.arange(num_timesteps).unsqueeze(0).repeat(batch_size, 1, 1).float() / num_timesteps,
        train_forces[batch_ids],
        train_img_feats[batch_ids],
        train_trajs[batch_ids]
    ], dim=-1).to(device)
    
    tar_x = torch.cat([
        torch.arange(num_timesteps).unsqueeze(0).repeat(batch_size, 1, 1).float() / num_timesteps,
        train_forces[batch_ids],
        train_img_feats[batch_ids],
    ], dim=-1).to(device)
    
    tar_y = train_trajs[batch_ids].to(device)
    obs_mask = torch.ones(batch_size, num_timesteps, dtype=torch.bool, device=device)
    tar_mask = torch.ones(batch_size, num_timesteps, dtype=torch.bool, device=device)
    
    # Forward pass
    optimizer.zero_grad()
    pred, gate = model(obs, tar_x, obs_mask)
    loss, nll = model.loss(pred, gate, tar_y, tar_mask)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    losses.append(nll.item())
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss = {nll.item():.4f}")

# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.show()

## Next Steps

This notebook showed a simplified example. For production training:

1. Use the full training scripts: `train_cnep_on_tactile.py` or `train_cnmp_on_tactile.py`
2. Implement proper validation
3. Use masked batching for variable-length observations
4. Save and load checkpoints
5. Monitor validation performance

See `TACTILE_TRAINING_GUIDE.md` for complete documentation.